# Lab 9: Analisi Documentale Agentica con GraphoLab

In questo notebook costruiamo una pipeline completa per analizzare un documento PDF contenente **testo, tabelle, figure e form** usando:

1. **PaddleOCR** per estrazione testo ordinato e layout detection
2. **qwen3-vl:8b** (Ollama locale) come VLM per tabelle e figure
3. **LangGraph** per orchestrare un agente che decide autonomamente quali strumenti usare
4. **GraphoLab core** — i moduli forensi già pronti (NER, firma, grafologia, dating)

Ispirato al notebook L6 del corso DeepLearning.AI *Document AI: From OCR to Agentic Doc Extraction*,
adattato per il contesto forense e con Ollama al posto di OpenAI.

---
### Architettura

```
Documento PDF
      │
      ├─► PaddleOCR (testo ordinato)          ─► contesto testuale
      ├─► PaddleOCR Layout Detection          ─► regioni (tabella, figura, testo)
      │         │
      │         └─► crop_region()             ─► immagini ritagliate
      │
      └─► LangGraph Agent (qwen3-vl:8b)
                │
                ├─► trascrivi_documento        ─► testo manoscritto
                ├─► estrai_entita             ─► persone, luoghi, org
                ├─► analizza_tabella          ─► dati strutturati (VLM)
                ├─► analizza_figura           ─► descrizione grafico (VLM)
                ├─► rileva_firma              ─► posizione firma
                ├─► data_documento            ─► date estratte
                └─► consulta_knowledge_base   ─► knowledge base forense
```

## Setup

Assicurati di avere Ollama in esecuzione con il modello scaricato:
```bash
ollama pull qwen3-vl:8b
ollama serve
```

In [ ]:
import os
import sys
from pathlib import Path

# Aggiungi la root di GraphoLab al path
ROOT = Path(".").parent.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"GraphoLab root: {ROOT}")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional

print("Librerie importate correttamente.")

---
## 1. Caricamento del documento PDF

Convertiamo il PDF in immagini (una per pagina) usando PyMuPDF.
Useremo la prima pagina per l'analisi, ma la pipeline può iterare su tutte.

In [ ]:
import fitz  # PyMuPDF

# ── Cambia questo path con il tuo documento ──────────────────────────────────
PDF_PATH = ROOT / "data" / "samples" / "documento_test.pdf"
# Se non hai un PDF, puoi usare un'immagine direttamente:
# IMAGE_PATH = ROOT / "data" / "samples" / "documento_test.jpg"
# ─────────────────────────────────────────────────────────────────────────────

def pdf_to_images(pdf_path: Path, dpi: int = 150) -> list:
    """Converti ogni pagina PDF in immagine numpy RGB."""
    doc = fitz.open(str(pdf_path))
    images = []
    zoom = dpi / 72
    mat = fitz.Matrix(zoom, zoom)
    for page in doc:
        pix = page.get_pixmap(matrix=mat)
        img = np.frombuffer(pix.samples, dtype=np.uint8)
        img = img.reshape(pix.height, pix.width, pix.n)
        if pix.n == 4:  # RGBA → RGB
            img = img[:, :, :3]
        images.append(img)
    doc.close()
    return images

if PDF_PATH.exists():
    pages = pdf_to_images(PDF_PATH)
    print(f"PDF caricato: {len(pages)} pagine")
    document_image = pages[0]
else:
    print(f"⚠️  File non trovato: {PDF_PATH}")
    print("Crea un PDF di test o modifica PDF_PATH con il percorso corretto.")
    # Crea immagine placeholder per continuare il notebook
    document_image = np.ones((1000, 800, 3), dtype=np.uint8) * 240

# Salva come file temporaneo per i tool che richiedono un path
import tempfile
tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
Image.fromarray(document_image).save(tmp.name)
DOCUMENT_IMAGE_PATH = tmp.name
print(f"Immagine documento salvata in: {DOCUMENT_IMAGE_PATH}")
print(f"Dimensioni: {document_image.shape[1]}x{document_image.shape[0]} px")

In [ ]:
# Visualizza il documento
plt.figure(figsize=(10, 14))
plt.imshow(document_image)
plt.axis("off")
plt.title("Documento caricato", fontsize=14)
plt.tight_layout()
plt.show()

---
## 2. Estrazione testo con PaddleOCR (ordinato per posizione)

Usiamo PaddleOCR per estrarre tutto il testo visibile nel documento,
ordinato dall'alto verso il basso e da sinistra a destra.

> **Differenza rispetto al notebook L6:** il notebook originale usava LayoutLMv3 (~500MB)
> per determinare l'ordine di lettura. Qui usiamo un sort per coordinata Y→X,
> sufficiente per documenti latini standard.

In [ ]:
from core.document_layout import extract_ordered_text

print("Estrazione testo in corso (primo avvio: scarica modelli PaddleOCR ~50MB)...")
ordered_text = extract_ordered_text(DOCUMENT_IMAGE_PATH)

print(f"\n{'='*60}")
print("TESTO ESTRATTO (ordinato per posizione)")
print('='*60)
print(ordered_text[:2000] + ("..." if len(ordered_text) > 2000 else ""))
print(f"\nTotale caratteri: {len(ordered_text)}")

In [ ]:
# Visualizza le bounding box del testo rilevato
from paddleocr import PaddleOCR

ocr = PaddleOCR(use_angle_cls=True, lang="it", show_log=False)
ocr_result = ocr.ocr(DOCUMENT_IMAGE_PATH, cls=True)

img_boxes = document_image.copy()
if ocr_result and ocr_result[0]:
    for line in ocr_result[0]:
        if line is None: continue
        pts = np.array(line[0], dtype=np.int32)
        cv2.polylines(img_boxes, [pts], True, (0, 120, 255), 1)

plt.figure(figsize=(10, 14))
plt.imshow(img_boxes)
plt.axis("off")
plt.title(f"Bounding box OCR ({len(ocr_result[0]) if ocr_result and ocr_result[0] else 0} righe rilevate)", fontsize=12)
plt.tight_layout()
plt.show()

---
## 3. Layout Detection — Regioni strutturate

PaddleOCR identifica automaticamente le regioni del documento:
- `text` — paragrafi di testo
- `table` — tabelle
- `figure` — grafici e immagini
- `title` — intestazioni

> Stesso approccio del notebook L6, sezione 2.

In [ ]:
from core.document_layout import detect_layout
from dataclasses import dataclass

@dataclass
class LayoutRegion:
    region_id: int
    region_type: str
    label: str
    bbox: list
    score: float

print("Layout detection in corso (primo avvio: scarica modello ~26MB)...")
layout_result = detect_layout(DOCUMENT_IMAGE_PATH)

layout_regions = [
    LayoutRegion(
        region_id=i,
        region_type=r["type"],
        label=r["label"],
        bbox=r["bbox"],
        score=r.get("score", 1.0),
    )
    for i, r in enumerate(layout_result.get("regions", []))
]

print(f"\nRegioni rilevate: {len(layout_regions)}")
for r in layout_regions:
    print(f"  [{r.region_id}] {r.label:12s}  confidenza: {r.score:.0%}  bbox: {r.bbox}")

In [ ]:
# Visualizza le regioni con colori per tipo
TYPE_COLORS = {
    "text":    "#4e9af1",
    "title":   "#f1c44e",
    "table":   "#4ef1a0",
    "figure":  "#f14e4e",
    "formula": "#c44ef1",
}

fig, ax = plt.subplots(figsize=(10, 14))
ax.imshow(document_image)

for r in layout_regions:
    x1, y1, x2, y2 = r.bbox
    color = TYPE_COLORS.get(r.region_type, "#aaaaaa")
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=2, edgecolor=color, facecolor=color, alpha=0.15
    )
    ax.add_patch(rect)
    ax.text(x1 + 4, y1 + 16, f"[{r.region_id}] {r.label}",
            color="white", fontsize=8, fontweight="bold",
            bbox=dict(facecolor=color, alpha=0.8, pad=1))

# Legenda
from matplotlib.patches import Patch
legend = [Patch(facecolor=c, label=t) for t, c in TYPE_COLORS.items()]
ax.legend(handles=legend, loc="upper right", fontsize=9)

ax.set_axis_off()
ax.set_title(f"Layout Detection — {len(layout_regions)} regioni", fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. Crop delle regioni di interesse

Ritagliamo le regioni tabella e figura per passarle al VLM.
Stesso approccio della sezione 2.4 del notebook L6.

In [ ]:
from core.document_layout import crop_region

tables  = [r for r in layout_regions if r.region_type in ("table", "tabella")]
figures = [r for r in layout_regions if r.region_type in ("figure", "figura", "chart", "image", "graph")]
texts   = [r for r in layout_regions if r.region_type in ("text", "title")]

print(f"Tabelle trovate:  {len(tables)}")
print(f"Figure trovate:   {len(figures)}")
print(f"Regioni testo:    {len(texts)}")

# Mostra crop di tabelle e figure
regions_to_show = tables + figures
if regions_to_show:
    ncols = min(3, len(regions_to_show))
    nrows = (len(regions_to_show) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
    axes = np.array(axes).flatten()
    for i, region in enumerate(regions_to_show):
        crop = crop_region(document_image, region.bbox, padding=8)
        axes[i].imshow(crop)
        axes[i].set_title(f"[{region.region_id}] {region.label}", fontsize=10)
        axes[i].axis("off")
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    plt.suptitle("Regioni ritagliate (tabelle + figure)", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("Nessuna tabella o figura rilevata nel documento.")

---
## 5. Analisi VLM — Tabelle e Figure con qwen3-vl:8b

Usiamo `call_vision_model()` per inviare le immagini ritagliate a **qwen3-vl:8b** via Ollama.

> **Differenza rispetto al notebook L6:** il notebook originale usava GPT-4o-mini (OpenAI, a pagamento).
> Qui usiamo qwen3-vl:8b in locale — stesso risultato, zero costi.

In [ ]:
from core.document_layout import call_vision_model
from core.rag import check_ollama

if not check_ollama():
    print("❌ Ollama non raggiungibile. Avvia con: ollama serve")
else:
    print("✅ Ollama raggiungibile")

In [ ]:
# ── Analisi tabelle ───────────────────────────────────────────────────────────
TABLE_PROMPT = """Sei un assistente forense. Analizza questa tabella estratta da un documento.
Estrai tutti i dati in formato Markdown. Mantieni tutte le righe e colonne.
Rispondi SOLO con la tabella Markdown, senza testo aggiuntivo."""

table_results = {}
for i, region in enumerate(tables):
    print(f"\nAnalisi tabella [{region.region_id}]...")
    crop = crop_region(document_image, region.bbox, padding=8)
    result = call_vision_model(crop, TABLE_PROMPT)
    table_results[region.region_id] = result
    print(result)

if not tables:
    print("Nessuna tabella da analizzare.")

In [ ]:
# ── Analisi figure/grafici ────────────────────────────────────────────────────
FIGURE_PROMPT = """Sei un assistente forense. Analizza questa figura estratta da un documento.
Descrivi in dettaglio: tipo di grafico, assi, valori, trend principali,
dati numerici visibili, legenda. Se è un'immagine generica, descrivila.
Rispondi in italiano in modo professionale."""

figure_results = {}
for i, region in enumerate(figures):
    print(f"\nAnalisi figura [{region.region_id}]...")
    crop = crop_region(document_image, region.bbox, padding=8)
    result = call_vision_model(crop, FIGURE_PROMPT)
    figure_results[region.region_id] = result
    print(result)

if not figures:
    print("Nessuna figura da analizzare.")

---
## 6. Analisi forense — Moduli GraphoLab

Applichiamo i moduli forensi specializzati di GraphoLab al documento.

In [ ]:
# ── 6.1 Named Entity Recognition ─────────────────────────────────────────────
from core.ner import ner_extract

if ordered_text.strip():
    print("Estrazione entità nominate...")
    entities, report_md = ner_extract(ordered_text)
    print(f"\nEntità trovate: {len(entities)}")
    print(report_md)
else:
    print("Testo non disponibile per NER.")

In [ ]:
# ── 6.2 Datazione documento ───────────────────────────────────────────────────
from core.dating import extract_dates

if ordered_text.strip():
    dates = extract_dates(ordered_text)
    if dates:
        print(f"Date trovate nel documento ({len(dates)}):")
        for raw, dt in dates:
            print(f"  '{raw}' → {dt.strftime('%d/%m/%Y')}")
    else:
        print("Nessuna data trovata nel testo.")

In [ ]:
# ── 6.3 Rilevamento firma ─────────────────────────────────────────────────────
from core.signature import sig_detect

print("Rilevamento firma in corso...")
annotated, summary = sig_detect(document_image, conf_threshold=0.3)
print(summary)

plt.figure(figsize=(10, 14))
plt.imshow(annotated)
plt.axis("off")
plt.title("Rilevamento firma (Conditional DETR)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.4 Analisi grafologica (se il documento contiene scrittura a mano) ───────
from core.graphology import grapho_analyse

print("Analisi grafologica...")
grapho_report, grapho_annotated = grapho_analyse(document_image)
print(grapho_report)

plt.figure(figsize=(10, 14))
plt.imshow(grapho_annotated)
plt.axis("off")
plt.title("Analisi grafologica — componenti rilevati", fontsize=12)
plt.tight_layout()
plt.show()

---
## 7. Agente LangGraph — Analisi completa in linguaggio naturale

Ora usiamo l'agente per fare domande libere sul documento.
L'agente decide autonomamente quali strumenti usare.

> Stesso approccio della sezione 4 del notebook L6, ma con:
> - LangGraph `create_react_agent` invece di `AgentExecutor` (obsoleto)
> - qwen3-vl:8b invece di GPT-4o-mini
> - 12 tool forensi invece di 2

In [ ]:
from core.agent import create_forensic_agent, agent_stream

# Verifica che l'agente si crei correttamente
agent = create_forensic_agent()
print("✅ Agente forense creato")
print(f"Modello: qwen3-vl:8b")
print(f"Tool disponibili: {len(agent.nodes)}")

In [ ]:
# Helper per eseguire l'agente e mostrare il ragionamento passo per passo
def run_agent(question: str, file_path: str = DOCUMENT_IMAGE_PATH):
    print(f"\n{'='*60}")
    print(f"DOMANDA: {question}")
    print('='*60)
    response = ""
    for update in agent_stream(question, [file_path], []):
        response = update  # accumulative
        # Mostra aggiornamenti intermedi (tool calls)
        if "🔧" in update and update != response:
            print(update.split("\n")[-1])  # ultima riga = azione corrente
    print("\nRISPOSTA FINALE:")
    # Rimuovi la sezione <details> per la stampa
    clean = response.split("<details>")[0].strip()
    print(clean)
    return response

### Test 1: Panoramica documento
Domanda generale — risponde dal testo OCR senza chiamare tool visivi.

In [ ]:
response1 = run_agent(
    "Di cosa parla questo documento? Identifica le persone nominate e la data."
)

### Test 2: Estrazione dati dalla tabella
L'agente chiama `analizza_tabella` → crop → qwen3-vl:8b → Markdown.

In [ ]:
response2 = run_agent(
    "Estrai e struttura i dati presenti nella tabella del documento."
)

### Test 3: Analisi figure e grafici
L'agente chiama `analizza_figura` → crop → qwen3-vl:8b → descrizione.

In [ ]:
response3 = run_agent(
    "Descrivi le figure o i grafici presenti nel documento e interpreta i dati mostrati."
)

### Test 4: Analisi firma forense
L'agente combina `rileva_firma` + `analisi_grafologica`.

In [ ]:
response4 = run_agent(
    "Rileva la firma nel documento e analizza le caratteristiche grafologiche della scrittura."
)

### Test 5: Analisi forense completa
L'agente orchestra più tool in sequenza autonomamente.

In [ ]:
response5 = run_agent(
    "Esegui un'analisi forense completa di questo documento: "
    "estrai il testo, identifica le persone, cerca date, "
    "rileva la firma, analizza tabelle e figure."
)

---
## 8. Riepilogo risultati

Raccogliamo tutti i risultati in un dizionario strutturato.

In [ ]:
import json
from datetime import datetime

report = {
    "documento": str(PDF_PATH),
    "timestamp": datetime.now().isoformat(),
    "testo_ocr": ordered_text[:500] + "..." if len(ordered_text) > 500 else ordered_text,
    "regioni_layout": [
        {"id": r.region_id, "tipo": r.label, "bbox": r.bbox}
        for r in layout_regions
    ],
    "tabelle": table_results,
    "figure": figure_results,
    "analisi_agente": {
        "panoramica": response1.split("<details>")[0].strip() if 'response1' in dir() else "",
        "tabelle": response2.split("<details>")[0].strip() if 'response2' in dir() else "",
        "figure": response3.split("<details>")[0].strip() if 'response3' in dir() else "",
        "firma": response4.split("<details>")[0].strip() if 'response4' in dir() else "",
        "forense_completa": response5.split("<details>")[0].strip() if 'response5' in dir() else "",
    }
}

print(json.dumps(report, indent=2, ensure_ascii=False)[:2000])

In [ ]:
# Salva il report in JSON
output_path = ROOT / "data" / "report_agentico.json"
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Report salvato in: {output_path}")

---
## Riepilogo

In questo notebook abbiamo costruito una pipeline di analisi documentale agentica che:

| Step | Tecnologia | Funzione GraphoLab |
|------|-----------|--------------------|
| 1. Caricamento PDF | PyMuPDF | — |
| 2. Estrazione testo | PaddleOCR | `extract_ordered_text()` |
| 3. Layout detection | PaddleOCR LayoutDetection | `detect_layout()` |
| 4. Crop regioni | PIL/NumPy | `crop_region()` |
| 5. Analisi tabelle | qwen3-vl:8b (Ollama) | `call_vision_model()` |
| 6. Analisi figure | qwen3-vl:8b (Ollama) | `call_vision_model()` |
| 7. NER | Wikineural multilingual | `ner_extract()` |
| 8. Firma | Conditional DETR | `sig_detect()` |
| 9. Grafologia | OpenCV features | `grapho_analyse()` |
| 10. Agente | LangGraph + qwen3-vl:8b | `agent_stream()` |

**Differenze rispetto al notebook L6 (DeepLearning.AI):**
- GPT-4o-mini → **qwen3-vl:8b** (locale, gratuito, privacy)
- `AgentExecutor` (obsoleto) → **`create_react_agent`** (LangGraph)
- 2 tool → **12 tool forensi** specializzati
- Contesto generico → **dominio forense** (firme, grafologia, ENFSI)